# Neural Network

In [1]:
import numpy as np
from sklearn.datasets import make_moons 
from sklearn.model_selection import train_test_split 
import matplotlib.pyplot as plt

In [2]:
nn_architecture = [
    {"input_dim": 2, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 6, "activation": "relu"},
    {"input_dim": 6, "output_dim": 6, "activation": "relu"}, 
    {"input_dim": 6, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 1, "activation": "sigmoid"},
]

In [3]:
def init_layers(nn_architecture, seed=99):
    np.random.seed(seed)
    params_values = {}

    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        params_values['W' + str(layer_idx)] = np.random.randn(
            layer["output_dim"], layer["input_dim"]) * 0.1
        
        params_values['b' + str(layer_idx)] = np.zeros((layer["output_dim"], 1))
        
    return params_values

In [6]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def sigmoid_backward(dA, x):
    sig = sigmoid(x)
    return dA * sig * (1-sig)

def relu_backward(dA, x):
    dx = np.array(dA, copy=True)
    dx[x <= 0] = 0
    return dx

In [9]:
def convert_prob_into_class(probs):
    """
    Converts probabilities into binary classes (0 or 1).
    """
    # Use 0.5 as the threshold for sigmoid outputs
    return (probs > 0.5).astype(int)


In [10]:
def single_layer_forward_propagation(A_prev, W_curr, b_curr, activation="relu"):
    # Step 1: Linear Transformation
    Z_curr = np.dot(W_curr, A_prev) + b_curr
    
    # Step 2: Selection of Activation Function
    if activation == "relu":
        activation_func = relu
    elif activation == "sigmoid":
        activation_func = sigmoid
    else:
        raise Exception("Non-supported activation function")
        
    # Step 3: Return both the activation (A) and the linear cache (Z)
    # The cache (Z) is crucial for backpropagation later!
    return activation_func(Z_curr), Z_curr


In [11]:
def get_cost_value(Y_hat, Y):
    # m is the number of examples
    m = Y_hat.shape[1]
    
    # Standard Cross-Entropy formula:
    # -1/m * (Y * log(Y_hat) + (1-Y) * log(1-Y_hat))
    cost = -1/m * (np.dot(Y, np.log(Y_hat).T) + np.dot(1 - Y, np.log(1 - Y_hat).T))
    
    # np.squeeze removes redundant dimensions (e.g., [[0.5]] becomes 0.5)
    return np.squeeze(cost)


In [12]:
def get_accuracy_value(Y_hat, Y):
    # Convert probabilities (0.0 to 1.0) into classes (0 or 1)
    Y_hat_class = convert_prob_into_class(Y_hat)
    
    # Compare predictions to actual labels and take the average
    return (Y_hat_class == Y).mean()


In [14]:
def single_layer_backward_propagation(dA_curr, W_curr, b_curr, Z_curr, A_prev, activation="relu"):
    m = A_prev.shape[1]
    if activation == "relu":
        backward_activation_func = relu_backward
    elif activation == "sigmoid":
        backward_activation_func = sigmoid_backward
    else:
        raise Exception('Non-supported activation function')
    
    dZ_curr = backward_activation_func(dA_curr, Z_curr)
    dW_curr = np.dot(dZ_curr, A_prev.T) / m
    db_curr = np.sum(dZ_curr, axis=1, keepdims=True) / m
    dA_prev = np.dot(W_curr.T, dZ_curr)
    return dA_prev, dW_curr, db_curr

In [3]:
def full_backward_propagation(Y_hat, Y, memory, params_values, nn_architecture):
    grads_values = {}
    Y = Y.reshape(Y_hat.shape)
    dA_prev = - (np.divide(Y, Y_hat) - np.divide(1 - Y, 1 - Y_hat))
    
    for layer_idx_prev, layer in reversed(list(enumerate(nn_architecture))):
        layer_idx_curr = layer_idx_prev + 1
        dA_prev, dW_curr, db_curr = single_layer_backward_propagation(
            dA_prev,
            params_values["W" + str(layer_idx_curr)],
            params_values["b" + str(layer_idx_curr)],
            memory["Z" + str(layer_idx_curr)],
            memory["A" + str(layer_idx_prev)],
            layer["activation"]
        )
        grads_values["dW" + str(layer_idx_curr)] = dW_curr
        grads_values["db" + str(layer_idx_curr)] = db_curr
    return grads_values